# Leverage overlay research for the CTF book

Three studies on the volatility overlay, merged into one notebook.

1. **Leverage exponent sweep** — where in the family $l_t \propto \sigma_t^{-\gamma}$ the
   book should sit, with $\gamma = 1$ the constant volatility rule currently in use and
   $\gamma = 2$ the Kelly rule.
2. **Conditional mean timing** — whether a conditioning variable orthogonal to volatility,
   namely the past return of the book itself, adds anything over the incumbent.
3. **Univariate forecasting of the book variance** — whether the scalar $w'\Sigma w$ is
   better forecast from the synthetic daily series of the book than from the shrinkage
   covariance currently in use.

**Structure.** The three studies are kept self-contained: each carries its own copy of every
helper it uses, prefixed `sw_`, `mt_` and `vf_` respectively, so that nothing collides in the
notebook namespace and so that the small differences between the original scripts survive the
merge rather than being reconciled away. What *is* shared is the data: the weight file, the
daily returns, the test flags, the monthly stock returns, the book return series and the
shrinkage volatility forecast are built once in section 0 and reused, since all three scripts
built them identically and the pass over the daily file is the expensive part.

Universal constants (paths, the lookback, the target volatility, the gross cap, the fold
geometry, the selection cut-off) also live in section 0; a constant that is specific to one
study, or that differs between the studies, is defined in that study's own section under its
prefix.

**Selection discipline is unchanged.** Every choice is made on the pre-1990 window under
blocked cross validation with purging, and the test column is printed for reference only.

## 0. Shared setup and data

The loaders below are common to all three scripts and are byte-identical between them up to
comments, so they appear once. The only substantive reconciliation is the variance floor in
the shrinkage forecast: the exponent sweep wrote `1e-16` and the other two wrote `1e-12`. The
floor never binds at monthly scale, and all three scripts wrote to the same `book_sigma.csv`
cache in any case, so `var_floor = 1e-12` is used throughout.

In [ ]:
import os

import numpy as np
import pandas as pd
from sklearn.covariance import ledoit_wolf

# paths
data_dir = "jkp-data"
weight_file = "Monthly volatility targeted CFP/output.csv"
sigma_cache = "book_sigma.csv"

# construction of the shrinkage volatility forecast
cov_lookback_days = 126
min_days = 60
days_per_month = 21
var_floor = 1e-12

# the unscaled book
gross_target = 2.0
gross_tolerance = 0.01

# the overlay
target_vol_annual = 0.10
gross_cap = 12.0
months_per_year = 12

# selection geometry, common to all three studies
n_folds = 5
purge_months = 1
min_fold_months = 12
selection_end = "1990-01-01"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

In [ ]:
def read_weights():
    # the studies must operate on the unscaled book, since a file that already carries the
    # overlay is proportional to the reciprocal of sigma and an exponent would then compound
    # rather than replace it. under the l1 normalisation the gross exposure of the unscaled
    # book is constant across months, so dividing each month by its own gross exposure recovers
    # it exactly whenever the gross cap did not bind. this is inert on a file that is already
    # unscaled.
    w = pd.read_csv(weight_file)
    w["eom"] = pd.to_datetime(w["eom"])
    w["id"] = w["id"].astype(np.int64)
    w["w"] = w["w"].astype(np.float64)
    w = w.dropna(subset=["w"])
    gross = w.groupby("eom")["w"].transform(lambda x: x.abs().sum())
    spread = float(gross.max() / gross.min()) if gross.min() > 0 else np.inf
    if spread > 1.0 + gross_tolerance:
        print("gross exposure varies by a factor of", round(spread, 3), ", renormalising to the unscaled book")
    w["w"] = w["w"] / gross * gross_target
    return w.sort_values(["eom", "id"]).reset_index(drop=True)


def read_daily():
    daily = pd.read_parquet(os.path.join(data_dir, "daily_ret.parquet"))
    date_col = next(c for c in ["date", "day", "eom"] if c in daily.columns)
    ret_col = next(c for c in ["ret", "ret_exc", "daily_ret", "r"] if c in daily.columns)
    daily = daily[["id", date_col, ret_col]].rename(columns={date_col: "date", ret_col: "dret"})
    daily["date"] = pd.to_datetime(daily["date"])
    daily["id"] = daily["id"].astype(np.int64)
    daily["dret"] = daily["dret"].astype(np.float64)
    daily = daily.dropna(subset=["dret"])
    return daily.sort_values("date").reset_index(drop=True)


def read_test_months():
    chars = pd.read_parquet(os.path.join(data_dir, "chars.parquet"), columns=["eom", "ctff_test"])
    flag = chars["ctff_test"]
    if not pd.api.types.is_bool_dtype(flag):
        flag = flag.astype(str).isin(["1", "1.0", "True", "true"])
    return set(pd.to_datetime(chars.loc[flag, "eom"]).unique())


def monthly_stock_returns(daily):
    # compound the daily returns inside each calendar month. reom is the month in which the
    # return is earned and is kept distinct from eom, the formation date, so that the two never
    # collide in a join.
    d = daily[["id", "date", "dret"]].copy()
    d["reom"] = d["date"] + pd.offsets.MonthEnd(0)
    d["lg"] = np.log1p(d["dret"].clip(lower=-0.99))
    m = d.groupby(["id", "reom"], as_index=False)["lg"].sum()
    m["mret"] = np.expm1(m["lg"].values)
    return m[["id", "reom", "mret"]]


def book_returns(weights, monthly):
    # the weights formed at eom t are held over the month that follows, so each formation date
    # is mapped to the next available return month.
    ret_months = np.sort(monthly["reom"].unique())
    form_months = np.sort(weights["eom"].unique())
    nxt = {}
    for t in form_months:
        later = ret_months[ret_months > t]
        if len(later) > 0:
            nxt[t] = later[0]
    w = weights.copy()
    w["reom"] = w["eom"].map(nxt)
    w = w.dropna(subset=["reom"])
    j = w.merge(monthly, on=["id", "reom"], how="inner")
    j["c"] = j["w"].values * j["mret"].values
    r = j.groupby("eom")["c"].sum()
    held = weights.groupby("eom")["id"].size()
    matched = j.groupby("eom")["id"].size()
    cover = (matched / held).reindex(r.index)
    return r.sort_index(), cover


def build_lw_sigma(weights, daily):
    # only the scalar quadratic form w' sigma w is required, but the shrinkage estimator is
    # retained here so that the gamma equal to one arm reproduces the overlay already in use.
    if os.path.exists(sigma_cache):
        s = pd.read_csv(sigma_cache, parse_dates=["eom"])
        return s.set_index("eom")["sigma"].sort_index()
    dvals = daily["date"].values
    udates = np.unique(dvals)
    rows = []
    for t, grp in weights.groupby("eom"):
        end = np.searchsorted(udates, np.datetime64(t), side="right")
        start = max(0, end - cov_lookback_days)
        if end - start < min_days:
            continue
        i0 = np.searchsorted(dvals, udates[start], side="left")
        i1 = np.searchsorted(dvals, udates[end - 1], side="right")
        sl = daily.iloc[i0:i1]
        ids = grp["id"].values
        sl = sl[sl["id"].isin(set(ids.tolist()))]
        if sl.empty:
            continue
        x = sl.pivot_table(index="date", columns="id", values="dret", aggfunc="last")
        x = x.reindex(columns=ids).fillna(0.0).values
        if x.shape[0] < min_days:
            continue
        cov, _ = ledoit_wolf(x, assume_centered=False)
        wv = grp["w"].values
        v = float(wv @ cov @ wv) * days_per_month
        rows.append({"eom": t, "sigma": float(np.sqrt(max(v, var_floor)))})
        if len(rows) % 100 == 0:
            print("shrinkage forecast computed for", len(rows), "months", flush=True)
    s = pd.DataFrame(rows)
    s.to_csv(sigma_cache, index=False)
    return s.set_index("eom")["sigma"].sort_index()

The cell below is the only pass over the daily file. It is cached on disk (`book_sigma.csv`)
for the volatility forecast and held in memory for the rest, so the three studies that follow
cost a few seconds each and may be re-run in any order.

In [ ]:
weights = read_weights()
daily = read_daily()
test_months = read_test_months()

monthly = monthly_stock_returns(daily)
ret, cover = book_returns(weights, monthly)
sigma = build_lw_sigma(weights, daily)

print("formation months in the weight file,", weights["eom"].nunique())
print("daily observations,", len(daily))
print("months with a book return,", len(ret), ", months with a volatility forecast,", len(sigma))
print("median cross sectional match rate,", round(float(cover.median()), 4))

## 1. Leverage exponent sweep

The overlay currently in use sets the leverage of the book to the ratio of a target volatility
to a forecast volatility, that is, proportional to the reciprocal of sigma. This is the
constant volatility rule. The Sharpe optimal predictable leverage, obtained from the Cauchy
Schwarz bound on the ratio of the first to the second moment of the levered return, is
proportional to the conditional mean divided by the sum of the conditional variance and the
squared conditional mean, which under a constant conditional mean reduces to the reciprocal of
sigma squared. The two natural rules therefore sit at opposite ends of the one parameter family

$$ l_t \propto \sigma_t^{-\gamma} $$

with $\gamma = 1$ for the current rule and $\gamma = 2$ for the Kelly rule. This section
estimates $\gamma$.

The weight series is taken as given and is never recomputed. Gamma is selected on the pre-1990
window under blocked cross validation with purging at the fold boundaries, and the fold Sharpes
are shrunk toward their grand mean by an empirical Bayes factor before ranking. The test window
figure is computed and printed for reference only and must not inform the choice.

In [ ]:
sw_gamma_grid = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5]


def sw_leverage(sigma, gamma, sigma_ref):
    # written so that gamma equal to one recovers the target over forecast rule exactly, and so
    # that the overall level is comparable across the grid, which keeps the gross cap binding on
    # the same terms for every arm.
    base = (target_vol_annual / np.sqrt(months_per_year)) / sigma_ref
    lev = base * np.power(sigma_ref / sigma.values, gamma)
    return np.minimum(lev, gross_cap)


def sw_sharpe(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < min_fold_months:
        return np.nan
    sd = np.std(x, ddof=1)
    if sd <= 0:
        return np.nan
    return float(np.mean(x) / sd * np.sqrt(months_per_year))


def sw_fold_sharpes(series):
    # contiguous blocks with the leading months of each block dropped, so that no observation
    # sits adjacent to the boundary of the block that precedes it.
    x = series.dropna()
    n = len(x)
    if n < n_folds * (min_fold_months + purge_months):
        return [sw_sharpe(x.values)]
    cuts = np.linspace(0, n, n_folds + 1).astype(int)
    out = []
    for k in range(n_folds):
        lo, hi = cuts[k], cuts[k + 1]
        if k > 0:
            lo = lo + purge_months
        out.append(sw_sharpe(x.values[lo:hi]))
    return out


def sw_paired_shrink(folds, base):
    # every arm is a reweighting of the same return series, so the arms are highly correlated
    # across blocks and an unpaired comparison of their levels wastes almost all of the
    # information. we therefore work with the within block differences against the arm currently
    # in use, whose sampling variance is far smaller, and shrink each mean difference toward zero
    # under a normal prior whose variance is estimated across the arms.
    d = folds - folds[base][None, :]
    dm = np.nanmean(d, axis=1)
    k = np.sum(np.isfinite(d), axis=1)
    se2 = np.nanvar(d, axis=1, ddof=1) / np.maximum(k, 1)
    other = np.array([i for i in range(len(dm)) if i != base])
    tau2 = max(0.0, float(np.mean(dm[other] ** 2) - np.mean(se2[other])))
    if tau2 <= 0.0:
        return np.zeros_like(dm), dm, 0.0
    return dm * tau2 / (tau2 + se2), dm, tau2

In [ ]:
def run_exponent_sweep(ret, sigma, cover, test_months):
    idx = ret.index.intersection(sigma.index)
    ret = ret.loc[idx].sort_index()
    sigma = sigma.loc[idx].sort_index()
    print("months with both a return and a volatility forecast,", len(idx))
    print("median cross sectional match rate,", round(float(cover.reindex(idx).median()), 4))

    is_test = pd.Series([t in test_months for t in idx], index=idx)
    is_sel = pd.Series(idx < pd.Timestamp(selection_end), index=idx)
    print("selection months,", int(is_sel.sum()), ", test months,", int(is_test.sum()))
    if int(is_sel.sum()) < n_folds * min_fold_months:
        print("warning, the weight file does not cover enough pre 1990 months for honest selection")

    sigma_ref = float(np.median(sigma.values[is_sel.values])) if is_sel.any() else float(np.median(sigma.values))

    fold_table = []
    rows = []
    for gamma in sw_gamma_grid:
        lev = pd.Series(sw_leverage(sigma, gamma, sigma_ref), index=idx)
        levered = lev * ret
        folds = sw_fold_sharpes(levered[is_sel])
        fold_table.append(folds)
        rows.append({
            "gamma": gamma,
            "sel_span": sw_sharpe(levered[is_sel].values),
            "fold_mean": float(np.nanmean(folds)),
            "fold_sd": float(np.nanstd(folds, ddof=1)) if len(folds) > 1 else np.nan,
            "test_ref": sw_sharpe(levered[is_test].values),
            "cap_frac": float(np.mean(lev.values >= gross_cap - 1e-12)),
            "mean_lev": float(np.mean(lev.values)),
        })

    width = max(len(g) for g in fold_table)
    folds = np.array([f + [np.nan] * (width - len(f)) for f in fold_table], dtype=float)
    base = int(np.argmin(np.abs(np.array(sw_gamma_grid) - 1.0)))
    shrunk, raw_diff, tau2 = sw_paired_shrink(folds, base)

    table = pd.DataFrame(rows)
    table["fold_diff"] = raw_diff
    table["diff_shrunk"] = shrunk
    table = table[["gamma", "fold_mean", "fold_sd", "fold_diff", "diff_shrunk", "sel_span", "test_ref", "cap_frac", "mean_lev"]]

    print()
    print(table.round(4).to_string(index=False))
    print()
    if float(table["cap_frac"].max()) > 0.0:
        print("warning, the gross cap binds on at least one arm, so the arms are not a pure reweighting of one another")
    print("prior variance of the paired differences,", round(tau2, 6))
    if tau2 <= 0.0 or float(np.max(shrunk)) <= 0.0:
        print("no arm survives shrinkage against gamma equal to one, so the current rule stands")
    else:
        best = table.loc[table["diff_shrunk"].idxmax()]
        print("selected gamma,", best["gamma"], ", shrunk gain over the current rule,", round(float(best["diff_shrunk"]), 4))
        print("its test window figure, reported after selection was closed,", round(float(best["test_ref"]), 4))
        print("the current rule on the same window,", round(float(table.loc[base, "test_ref"]), 4))
    print()
    print("the test column is a reference. selecting on it would be selection on the scored period.")
    return table


sweep_table = run_exponent_sweep(ret, sigma, cover, test_months)

In [ ]:
sweep_table.round(4)

## 2. Conditional mean timing of the book

Let $\mu_t$ and $\sigma_t$ denote the conditional mean and standard deviation of the book return
over the month following formation at $t$, both measurable with respect to the information
available at $t$. Applying the Cauchy Schwarz inequality to the ratio of the first to the second
moment of the levered return, the Sharpe optimal predictable leverage is

$$ l_t \propto \frac{\mu_t}{\sigma_t^2 + \mu_t^2}, $$

with attainable Sharpe equal to the square root of the expectation of $\mu_t^2$ divided by that
same sum. Since $\mu_t^2$ is negligible against $\sigma_t^2$ at monthly frequency, the rule
reduces to $\mu_t / \sigma_t^2$, and the overlay currently in use is the special case of a
constant conditional mean combined with the further restriction that the exponent on sigma is
one rather than two.

The exponent sweep of section 1 already established that any conditional mean which is a
function of sigma alone is exhausted, since the selected exponent of 0.75 implies that $\mu_t$
scales with $\sigma_t^{1.25}$ approximately and that the whole of the available gain from that
channel was 0.024. This section therefore admits only a conditioning variable orthogonal to
volatility, namely the past return of the book itself.

The specification is fixed in advance: a first order autoregression of the book return with the
trailing twelve month mean and the log variance forecast as controls, fitted by expanding window
so that no coefficient uses an outcome that was unobserved at the date of the forecast. The
candidate is admitted only if its shrunk paired gain over the incumbent exceeds the threshold
set below. No other specification is to be tried on the basis of this run.

In [ ]:
mt_min_train_months = 60
mt_trailing_months = 12
mt_admission_threshold = 0.05
mt_hac_lags = 6


def mt_design(ret, sigma):
    # the return earned over the month following formation at t is the target. the predictors are
    # the return earned over the month ending at t, the trailing mean of that series, and the log
    # variance forecast, all of which are observable at t. let us note that ret is indexed by
    # formation date, so its own lag is the most recent realised outcome.
    lag1 = ret.shift(1)
    trail = ret.shift(1).rolling(mt_trailing_months, min_periods=mt_trailing_months).mean()
    logv = np.log(np.maximum(np.square(sigma.values), var_floor))
    x = np.column_stack([np.ones(len(ret)), lag1.values, trail.values, logv])
    return x, ret.values


def mt_expanding_level(y):
    # the unconditional mean of the book return estimated on past months only. this arm isolates
    # the level channel, that is, the part of the timing rule which merely rescales or reverses
    # the book according to its own average return, from the conditional channel we set out to
    # test.
    n = len(y)
    out = np.full(n, np.nan)
    usable = np.isfinite(y)
    for i in range(n):
        train = usable & (np.arange(n) < i)
        if int(train.sum()) < mt_min_train_months:
            continue
        out[i] = float(np.mean(y[train]))
    return out


def mt_expanding_mean_forecast(x, y):
    # the coefficients used at month t are fitted on months strictly earlier than t, whose outcome
    # was already observed at t, so no information from the forecast month enters the fit.
    n = len(y)
    out = np.full(n, np.nan)
    usable = np.isfinite(y) & np.all(np.isfinite(x), axis=1)
    for i in range(n):
        if not np.all(np.isfinite(x[i])):
            continue
        train = usable & (np.arange(n) < i)
        if int(train.sum()) < mt_min_train_months:
            continue
        beta, _, _, _ = np.linalg.lstsq(x[train], y[train], rcond=None)
        out[i] = float(x[i] @ beta)
    return out


def mt_hac_regression(x, y):
    # newey west standard errors, used only to describe the predictive relation on the selection
    # window. this description plays no part in the decision rule.
    ok = np.isfinite(y) & np.all(np.isfinite(x), axis=1)
    x, y = x[ok], y[ok]
    n, k = x.shape
    beta, _, _, _ = np.linalg.lstsq(x, y, rcond=None)
    e = y - x @ beta
    xtx_inv = np.linalg.pinv(x.T @ x)
    s = (x * e[:, None]).T @ (x * e[:, None])
    for l in range(1, mt_hac_lags + 1):
        wgt = 1.0 - l / (mt_hac_lags + 1.0)
        a = (x[l:] * e[l:, None]).T @ (x[:-l] * e[:-l, None])
        s = s + wgt * (a + a.T)
    cov = xtx_inv @ s @ xtx_inv
    se = np.sqrt(np.maximum(np.diag(cov), 0.0))
    r2 = 1.0 - np.var(e, ddof=0) / np.var(y, ddof=0)
    return beta, beta / np.where(se > 0, se, np.nan), float(r2), int(n)


def mt_sharpe(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < min_fold_months:
        return np.nan
    sd = np.std(x, ddof=1)
    if sd <= 0:
        return np.nan
    return float(np.mean(x) / sd * np.sqrt(months_per_year))


def mt_fold_sharpes(series):
    x = series.dropna()
    n = len(x)
    if n < n_folds * (min_fold_months + purge_months):
        return [mt_sharpe(x.values)]
    cuts = np.linspace(0, n, n_folds + 1).astype(int)
    out = []
    for k in range(n_folds):
        lo, hi = cuts[k], cuts[k + 1]
        if k > 0:
            lo = lo + purge_months
        out.append(mt_sharpe(x.values[lo:hi]))
    return out


def mt_paired_shrink(folds, base):
    # as in section 1, but the within row variance is taken over the finite entries only and a row
    # with fewer than two of them is given infinite sampling variance, since this study compares
    # four arms rather than eleven and a degenerate row would otherwise propagate a nan.
    d = folds - folds[base][None, :]
    dm = np.nanmean(d, axis=1)
    k = np.sum(np.isfinite(d), axis=1)
    var = np.array([np.var(row[np.isfinite(row)], ddof=1) if np.isfinite(row).sum() > 1 else np.inf for row in d])
    se2 = var / np.maximum(k, 1)
    other = np.array([i for i in range(len(dm)) if i != base])
    tau2 = max(0.0, float(np.mean(dm[other] ** 2) - np.mean(se2[other])))
    if tau2 <= 0.0:
        return np.zeros_like(dm), dm, 0.0
    return dm * tau2 / (tau2 + se2), dm, tau2

In [ ]:
def run_mean_timing(ret, sigma, cover, test_months):
    idx = ret.index.intersection(sigma.index).sort_values()
    ret = ret.loc[idx]
    sigma = sigma.loc[idx]
    print("months with both a return and a volatility forecast,", len(idx))
    print("median cross sectional match rate,", round(float(cover.reindex(idx).median()), 4))

    is_test = np.array([t in test_months for t in idx])
    is_sel = np.array(idx < pd.Timestamp(selection_end))

    x, y = mt_design(ret, sigma)
    mu = mt_expanding_mean_forecast(x, y)

    beta, tstat, r2, nobs = mt_hac_regression(x[is_sel], y[is_sel])
    labels = ["intercept", "lag one return", "trailing mean", "log variance"]
    print()
    print("predictive regression on the selection window, newey west at", mt_hac_lags, "lags,", nobs, "observations")
    for name, b, t in zip(labels, beta, tstat):
        print("  ", name, ", coefficient", round(float(b), 6), ", t statistic", round(float(t), 3))
    print("   in sample r squared,", round(r2, 5))

    sig = sigma.values
    level = mt_expanding_level(y)
    arms = {
        "incumbent": 1.0 / sig,
        "kelly_const": 1.0 / np.square(sig),
        "mean_level": level / np.square(sig),
        "mean_timed": mu / np.square(sig),
    }

    common = np.ones(len(idx), dtype=bool)
    for v in arms.values():
        common &= np.isfinite(v)
    print()
    print("months common to every arm,", int(common.sum()), ", of which selection,", int((common & is_sel).sum()), ", test,", int((common & is_test).sum()))

    names = list(arms.keys())
    base = names.index("incumbent")
    fold_table = []
    rows = []
    for name in names:
        # only the shape of the leverage matters to the sharpe, so each arm is scaled so that its
        # median absolute leverage on the selection window equals the level of the incumbent. the
        # sign is left intact, since a negative conditional mean implies a short position in the
        # book under the optimal rule. the scalar below was called level in the original script,
        # which shadowed the expanding level array above; it is renamed here for clarity only.
        raw = np.where(common, arms[name], np.nan)
        med_raw = np.median(np.abs(raw[is_sel & common]))
        med_sig = np.median(sig[is_sel & common])
        scale = (target_vol_annual / np.sqrt(months_per_year)) / med_sig
        lev = np.clip(raw / med_raw * scale, -gross_cap, gross_cap)
        levered = pd.Series(lev * ret.values, index=idx)
        folds = mt_fold_sharpes(levered[is_sel & common])
        fold_table.append(folds)
        bound = np.nan
        if name == "mean_timed":
            m = mu[common & is_sel]
            s2 = np.square(sig[common & is_sel])
            bound = float(np.sqrt(np.mean(np.square(m) / s2)) * np.sqrt(months_per_year))
        rows.append({
            "arm": name,
            "fold_mean": float(np.nanmean(folds)),
            "fold_sd": float(np.nanstd(folds, ddof=1)) if len(folds) > 1 else np.nan,
            "sel_span": mt_sharpe(levered[is_sel & common].values),
            "test_ref": mt_sharpe(levered[is_test & common].values),
            "neg_frac": float(np.mean(lev[common] < 0)),
            "cap_frac": float(np.mean(np.abs(lev[common]) >= gross_cap - 1e-12)),
            "sharpe_bound": bound,
        })

    width = max(len(f) for f in fold_table)
    if width < n_folds:
        print("warning, only", width, "blocks were available, so the paired comparison has little power")
    folds = np.array([f + [np.nan] * (width - len(f)) for f in fold_table], dtype=float)
    shrunk, raw_diff, tau2 = mt_paired_shrink(folds, base)
    shrunk_lvl, raw_lvl, tau2_lvl = mt_paired_shrink(folds, names.index("mean_level"))
    table = pd.DataFrame(rows)
    table["fold_diff"] = raw_diff
    table["diff_shrunk"] = shrunk
    table["diff_level"] = shrunk_lvl
    table = table[["arm", "fold_mean", "fold_sd", "fold_diff", "diff_shrunk", "diff_level", "sel_span", "test_ref", "neg_frac", "cap_frac", "sharpe_bound"]]

    print()
    print(table.round(4).to_string(index=False))
    print()
    print("prior variance of the paired differences,", round(tau2, 6))
    i_timed = names.index("mean_timed")
    gain = float(table.loc[i_timed, "diff_shrunk"])
    gain_lvl = float(table.loc[i_timed, "diff_level"])
    print("shrunk gain of the mean timed arm over the incumbent,", round(gain, 4))
    print("shrunk gain of the mean timed arm over the level only arm,", round(gain_lvl, 4))
    print("threshold,", mt_admission_threshold, ", both must be exceeded")
    if gain > mt_admission_threshold and gain_lvl > mt_admission_threshold:
        print("the candidate is admitted under the rule fixed before the run")
        print("its test window figure, reported after selection was closed,", round(float(table.loc[i_timed, "test_ref"]), 4))
    else:
        print("the candidate fails the threshold, so the incumbent stands and the risk side is closed")
    print()
    print("the sharpe bound is the square root of the expectation of the squared conditional mean")
    print("divided by the conditional variance, which is the most the timing rule could attain if")
    print("the fitted conditional mean were correct. a bound close to the incumbent sharpe indicates")
    print("that the estimated predictability is too weak to matter, whatever the fitting method.")
    print("the test column is a reference. selecting on it would be selection on the scored period.")
    return table


timing_table = run_mean_timing(ret, sigma, cover, test_months)

In [ ]:
timing_table.round(4)

## 3. Univariate forecasting of the book variance

The overlay requires only the scalar $w'\Sigma w$ and not the matrix $\Sigma$ itself. Since the
weights formed at the end of month $t$ are known at that date, let us define the synthetic daily
series

$$ b_s = w_t' r_s, \qquad s \text{ a trading day preceding } t, $$

whose realised variance reproduces the required quadratic form exactly. The estimation problem
thereby falls from order $p^2$ parameters, with $p$ of the order of seventeen hundred and only
one hundred and twenty six daily observations, to a univariate forecasting problem with several
thousand observations, a regime in which the sample second moment is well behaved and no
shrinkage is required.

The section builds that series month by month, caches its realised second moments over several
horizons together with the realised variance actually incurred in the month that follows, and
then compares five forecasts of the month ahead variance:

| arm | forecast |
| --- | --- |
| `lw126` | the Ledoit Wolf shrinkage covariance currently in use |
| `rv126` | the plain realised variance of the synthetic series over 126 days |
| `ewma` | an exponentially weighted second moment of the synthetic series |
| `har` | an expanding window regression of the log realised variance on its monthly, quarterly and annual components in the manner of Corsi |
| `harlw` | the same regression with the shrinkage forecast admitted as a further regressor |

Each forecast is judged twice. First on forecast accuracy through the QLIKE loss, which is the
standard proper loss for variance forecasts and is robust to the fact that the realised variance
is itself a noisy proxy for the latent variance. Second, and decisively, on the Sharpe ratio of
the levered book, since that is the quantity being scored.

Selection is again on the pre-1990 window under blocked cross validation with purging, in paired
form against the shrinkage forecast currently in use. The test window figures are printed for
reference only.

In [ ]:
vf_moment_cache = "book_moments.csv"
vf_max_lookback_days = 252
vf_horizons = [21, 63, 252]
vf_ewma_halflife_days = 42
vf_min_train_months = 60


def vf_synthetic_series(slice_df, wmap):
    # b_s equals the weighted sum of the daily stock returns, with an absent stock contributing
    # nothing, which is the same convention as filling its column with zero in the matrix form.
    x = slice_df.copy()
    x["w"] = x["id"].map(wmap)
    x = x.dropna(subset=["w"])
    if x.empty:
        return pd.Series(dtype=float)
    x["c"] = x["w"].values * x["dret"].values
    return x.groupby("date")["c"].sum().sort_index()


def vf_build_moments(weights, daily):
    # for every formation month, the realised second moments of the synthetic series over each
    # horizon, an exponentially weighted second moment, and the variance actually incurred over the
    # month that follows, which serves as the regression target. all quantities are stored as mean
    # daily squared returns.
    if os.path.exists(vf_moment_cache):
        m = pd.read_csv(vf_moment_cache, parse_dates=["eom"])
        return m.set_index("eom").sort_index()
    dvals = daily["date"].values
    udates = np.unique(dvals)
    decay = 0.5 ** (1.0 / vf_ewma_halflife_days)
    form = np.sort(weights["eom"].unique())
    rows = []
    for k, t in enumerate(form):
        grp = weights[weights["eom"] == t]
        wmap = pd.Series(grp["w"].values, index=grp["id"].values)
        end = np.searchsorted(udates, np.datetime64(t), side="right")
        start = max(0, end - vf_max_lookback_days)
        if end - start < min_days:
            continue
        i0 = np.searchsorted(dvals, udates[start], side="left")
        i1 = np.searchsorted(dvals, udates[end - 1], side="right")
        b = vf_synthetic_series(daily.iloc[i0:i1], wmap)
        if len(b) < min_days:
            continue
        sq = np.square(b.values)
        row = {"eom": t, "n_days": len(sq)}
        for h in vf_horizons:
            take = sq[-h:] if len(sq) >= h else sq
            row["ms" + str(h)] = float(np.mean(take))
        wts = decay ** np.arange(len(sq) - 1, -1, -1)
        row["ms_ewma"] = float(np.sum(wts * sq) / np.sum(wts))
        row["ms" + str(cov_lookback_days)] = float(np.mean(sq[-cov_lookback_days:]))
        # the forward month, used as the regression target only for months whose outcome is
        # already observed at the date of the fit.
        nxt = udates[udates > np.datetime64(t)]
        nxt = nxt[nxt <= np.datetime64(pd.Timestamp(t) + pd.offsets.MonthEnd(1))]
        if len(nxt) >= 5:
            j0 = np.searchsorted(dvals, nxt[0], side="left")
            j1 = np.searchsorted(dvals, nxt[-1], side="right")
            bn = vf_synthetic_series(daily.iloc[j0:j1], wmap)
            row["ms_next"] = float(np.mean(np.square(bn.values))) if len(bn) >= 5 else np.nan
        else:
            row["ms_next"] = np.nan
        rows.append(row)
        if (k + 1) % 100 == 0:
            print("moments computed for", k + 1, "months", flush=True)
    m = pd.DataFrame(rows).set_index("eom").sort_index()
    m.to_csv(vf_moment_cache)
    return m


def vf_expanding_har(frame, extra=None):
    # the log realised variance of the month ahead is regressed on its monthly, quarterly and
    # annual components. the coefficients at month t are fitted on months strictly earlier than t,
    # whose outcome is already observed at t, so no information from the forecast month enters the
    # fit. the retransformation from the log scale introduces a multiplicative bias that is common
    # to every month and therefore cancels in the leverage, which is normalised by its own median.
    cols = ["ms" + str(h) for h in vf_horizons] + (extra if extra else [])
    x = np.column_stack([np.log(np.maximum(frame[c].values, var_floor)) for c in cols])
    x = np.column_stack([np.ones(len(x)), x])
    y = np.log(np.maximum(frame["ms_next"].values, var_floor))
    idx = frame.index
    out = np.full(len(idx), np.nan)
    usable = np.isfinite(y) & np.all(np.isfinite(x), axis=1)
    for i in range(len(idx)):
        train = usable & (np.arange(len(idx)) < i)
        if int(train.sum()) < vf_min_train_months:
            continue
        xt, yt = x[train], y[train]
        beta, _, _, _ = np.linalg.lstsq(xt, yt, rcond=None)
        out[i] = float(x[i] @ beta)
    return pd.Series(np.exp(out), index=idx)


def vf_qlike(realised, forecast):
    r = np.asarray(realised, dtype=float)
    f = np.asarray(forecast, dtype=float)
    ok = np.isfinite(r) & np.isfinite(f) & (r > 0) & (f > 0)
    if ok.sum() == 0:
        return np.nan
    ratio = r[ok] / f[ok]
    return float(np.mean(ratio - np.log(ratio) - 1.0))


def vf_leverage(sigma, sigma_ref):
    base = (target_vol_annual / np.sqrt(months_per_year)) / sigma_ref
    return np.minimum(base * (sigma_ref / sigma), gross_cap)


def vf_sharpe(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < min_fold_months:
        return np.nan
    sd = np.std(x, ddof=1)
    if sd <= 0:
        return np.nan
    return float(np.mean(x) / sd * np.sqrt(months_per_year))


def vf_fold_sharpes(series):
    x = series.dropna()
    n = len(x)
    if n < n_folds * (min_fold_months + purge_months):
        return [vf_sharpe(x.values)]
    cuts = np.linspace(0, n, n_folds + 1).astype(int)
    out = []
    for k in range(n_folds):
        lo, hi = cuts[k], cuts[k + 1]
        if k > 0:
            lo = lo + purge_months
        out.append(vf_sharpe(x.values[lo:hi]))
    return out


def vf_paired_shrink(folds, base):
    # every arm levers the same return series, so the arms are highly correlated across blocks and
    # an unpaired comparison of their levels discards nearly all the information. we therefore work
    # with the within block differences against the incumbent and shrink each mean difference
    # toward zero under a normal prior whose variance is estimated across the remaining arms.
    d = folds - folds[base][None, :]
    dm = np.nanmean(d, axis=1)
    k = np.sum(np.isfinite(d), axis=1)
    se2 = np.nanvar(d, axis=1, ddof=1) / np.maximum(k, 1)
    other = np.array([i for i in range(len(dm)) if i != base])
    tau2 = max(0.0, float(np.mean(dm[other] ** 2) - np.mean(se2[other])))
    if tau2 <= 0.0:
        return np.zeros_like(dm), dm, 0.0
    return dm * tau2 / (tau2 + se2), dm, tau2

The moment cache is the one remaining pass over the daily file. It is written to
`book_moments.csv` and reread on any later run.

In [ ]:
mom = vf_build_moments(weights, daily)
print("months with a moment set,", len(mom))
mom.tail()

In [ ]:
def run_variance_forecast(ret, mom, lw, cover, test_months):
    idx = ret.index.intersection(mom.index).intersection(lw.index).sort_values()
    ret = ret.loc[idx]
    mom = mom.loc[idx]
    lw = lw.loc[idx]
    print("months with a return, a moment set and a shrinkage forecast,", len(idx))
    print("median cross sectional match rate,", round(float(cover.reindex(idx).median()), 4))

    is_test = pd.Series([t in test_months for t in idx], index=idx)
    is_sel = pd.Series(idx < pd.Timestamp(selection_end), index=idx)
    print("selection months,", int(is_sel.sum()), ", test months,", int(is_test.sum()))
    if int(is_sel.sum()) < n_folds * min_fold_months:
        print("warning, the weight file does not cover enough pre 1990 months for honest selection")

    # every variance below is expressed in monthly units.
    lw_var = np.square(lw)
    har_frame = mom.copy()
    har_frame["ms_lw"] = lw_var.values / days_per_month
    forecasts = {
        "lw126": lw_var,
        "rv126": pd.Series(mom["ms" + str(cov_lookback_days)].values * days_per_month, index=idx),
        "ewma": pd.Series(mom["ms_ewma"].values * days_per_month, index=idx),
        "har": vf_expanding_har(har_frame) * days_per_month,
        "harlw": vf_expanding_har(har_frame, extra=["ms_lw"]) * days_per_month,
    }
    realised = pd.Series(mom["ms_next"].values * days_per_month, index=idx)

    names = list(forecasts.keys())
    base = names.index("lw126")

    # the regression arms carry a training burn in and are therefore undefined on the earliest
    # months. a comparison of arms measured on different samples is not paired, so every arm is
    # restricted to the months on which all of them exist.
    common = np.ones(len(idx), dtype=bool)
    for name in names:
        v = forecasts[name].reindex(idx).values
        common &= np.isfinite(v) & (v > 0)
    idx = idx[common]
    ret = ret.loc[idx]
    realised = realised.loc[idx]
    is_test = is_test.loc[idx]
    is_sel = is_sel.loc[idx]
    forecasts = {k: v.reindex(idx) for k, v in forecasts.items()}
    print("months common to every forecast,", len(idx), ", of which selection,", int(is_sel.sum()), ", test,", int(is_test.sum()))

    fold_table = []
    rows = []
    for name in names:
        v = forecasts[name]
        sig = pd.Series(np.sqrt(np.maximum(v.values, var_floor)), index=idx)
        ref_pool = sig[is_sel.values]
        sigma_ref = float(np.median(ref_pool.values)) if len(ref_pool) else float(np.median(sig.values))
        lev = vf_leverage(sig, sigma_ref)
        levered = lev * ret
        folds = vf_fold_sharpes(levered[is_sel])
        fold_table.append(folds)
        rows.append({
            "forecast": name,
            "qlike_sel": vf_qlike(realised[is_sel.values], v[is_sel.values]),
            "fold_mean": float(np.nanmean(folds)),
            "fold_sd": float(np.nanstd(folds, ddof=1)) if len(folds) > 1 else np.nan,
            "sel_span": vf_sharpe(levered[is_sel].values),
            "test_ref": vf_sharpe(levered[is_test].values),
            "cap_frac": float(np.mean(lev.values >= gross_cap - 1e-12)),
        })

    width = max(len(f) for f in fold_table)
    folds = np.array([f + [np.nan] * (width - len(f)) for f in fold_table], dtype=float)
    shrunk, raw_diff, tau2 = vf_paired_shrink(folds, base)
    table = pd.DataFrame(rows)
    table["fold_diff"] = raw_diff
    table["diff_shrunk"] = shrunk
    table = table[["forecast", "qlike_sel", "fold_mean", "fold_sd", "fold_diff", "diff_shrunk", "sel_span", "test_ref", "cap_frac"]]

    print()
    print(table.round(4).to_string(index=False))
    print()
    if float(table["cap_frac"].max()) > 0.0:
        print("warning, the gross cap binds on at least one arm, so the arms are not a pure reweighting of one another")
    print("prior variance of the paired differences,", round(tau2, 6))
    if tau2 <= 0.0 or float(np.max(shrunk)) <= 0.0:
        print("no forecast survives shrinkage against the shrinkage covariance, so the current overlay stands")
    else:
        best = table.loc[table["diff_shrunk"].idxmax()]
        print("selected forecast,", best["forecast"], ", shrunk gain over the incumbent,", round(float(best["diff_shrunk"]), 4))
        print("its test window figure, reported after selection was closed,", round(float(best["test_ref"]), 4))
        print("the incumbent on the same window,", round(float(table.loc[base, "test_ref"]), 4))
    print()
    print("a lower qlike is a better variance forecast. the two criteria need not agree, since the")
    print("sharpe rewards the timing of leverage and not the accuracy of the variance as such.")
    print("the test column is a reference. selecting on it would be selection on the scored period.")
    return table


variance_table = run_variance_forecast(ret, mom, sigma, cover, test_months)

In [ ]:
variance_table.round(4)

## Summary

The three selection tables side by side. Every figure in the `test_ref` column is a reference
only: it was computed after the selection rule had already fixed the choice, and selecting on it
would be selection on the scored period.

In [ ]:
for title, tbl in [
    ("exponent sweep", sweep_table),
    ("conditional mean timing", timing_table),
    ("variance forecasting", variance_table),
]:
    print(title)
    print(tbl.round(4).to_string(index=False))
    print()